# Notebook 1: From Source Metadata to Ontology

Every knowledge graph starts with a question: **where does the ontology come from?**

In practice, most organizations already have their domain knowledge encoded - in relational schemas, data dictionaries, API specs, or documentation. The process of deriving an ontology from these sources is called **ontology induction**.

This notebook walks through that journey for our insurance domain:
1. Inspect the relational schema (the source of truth)
2. Identify entities, relationships, and constraints
3. See how these map to an OWL ontology
4. Explore the resulting ontology

> **Note:** Full ontology induction - including automated extraction from schemas, NLP-based concept discovery, and ontology learning from text - is a rich research area beyond the scope of this workshop. Here we focus on the manual, schema-driven approach that's most common in enterprise settings.


## The Relational Schema

Our insurance data lives in 5 relational tables. This is the SQL DDL that creates them:

```sql
CREATE TABLE customer (
  id VARCHAR(20) PRIMARY KEY,
  customer_id VARCHAR(20) NOT NULL,
  customer_name VARCHAR(100) NOT NULL
);

CREATE TABLE policy (
  id VARCHAR(10) PRIMARY KEY,
  policy_id VARCHAR(20) NOT NULL,
  policy_type VARCHAR(10) NOT NULL,
  customer_id VARCHAR(20) NOT NULL REFERENCES customer(id)
);

CREATE TABLE policy_coverage (
  id VARCHAR(20) PRIMARY KEY,
  policy_id VARCHAR(10) NOT NULL REFERENCES policy(id),
  coverage_amount DECIMAL(12,2) NOT NULL,
  deductible DECIMAL(12,2) NOT NULL,
  premium DECIMAL(12,2) NOT NULL,
  valid_from TIMESTAMP NOT NULL,
  valid_to TIMESTAMP,
  is_current BOOLEAN NOT NULL
);

CREATE TABLE claim (
  id VARCHAR(10) PRIMARY KEY,
  claim_id VARCHAR(20) NOT NULL,
  claim_amount DECIMAL(12,2) NOT NULL,
  incident_date TIMESTAMP NOT NULL,
  policy_id VARCHAR(10) NOT NULL REFERENCES policy(id)
);

CREATE TABLE claim_event (
  id VARCHAR(20) PRIMARY KEY,
  claim_id VARCHAR(10) NOT NULL REFERENCES claim(id),
  event_type VARCHAR(20) NOT NULL,
  event_timestamp TIMESTAMP NOT NULL,
  event_amount DECIMAL(12,2)
);
```

### What can we infer from this schema?

| Schema Element | Ontology Concept |
|---------------|-----------------|
| Tables | → **Classes** (Customer, Policy, PolicyCoverage, Claim, ClaimEvent) |
| Columns | → **Datatype Properties** (claimAmount, incidentDate, policyType, ...) |
| Foreign keys | → **Object Properties** (forPolicy, hasCoverage, hasEvent, heldBy) |
| NOT NULL constraints | → **Cardinality constraints** (sh:minCount 1) |
| REFERENCES | → **Domain/Range** declarations |
| `valid_from`/`valid_to` pattern | → **SCD Type 2** temporal modeling |
| `event_type` allowed values | → **Enumeration constraints** (sh:in) |

This mapping from relational metadata to ontology constructs is the core of schema-driven ontology induction.


## Ontology Induction Approaches

In practice, ontologies are built through a combination of:

| Approach | Description | When to Use |
|----------|-------------|-------------|
| **Schema-driven** | Map tables → classes, columns → properties, FKs → relationships | You have a well-designed relational schema |
| **Top-down** | Start from domain expertise, model concepts first | Greenfield domain modeling |
| **NLP-based** | Extract concepts and relationships from text corpora | Large unstructured documentation |
| **Ontology reuse** | Import from existing ontologies (Schema.org, FIBO, etc.) | Standard domains with published ontologies |
| **LLM-assisted** | Use LLMs to propose class/property structures from descriptions | Rapid prototyping, then human refinement |

For this workshop, we used the **schema-driven** approach - the relational schema above was our starting point.

> **Ontology validation** is equally important: techniques like competency questions ("Can the ontology answer X?"), SHACL constraint checking, and consistency reasoning (OWL DL) help ensure the ontology is correct and complete. We'll see SHACL validation in Notebook 02.


## The Resulting Ontology

Here's the OWL ontology that was derived from the relational schema above. Notice the direct correspondence between tables/columns and classes/properties:


In [ ]:
from workshop import *

Let's examine the raw Turtle file. This is the complete ontology - every class, property, and relationship that defines our insurance domain:


In [ ]:
# Display the ontology TTL with syntax highlighting
ontology_ttl = (ONTOLOGY_DIR / "ontology.ttl").read_text()
display(Markdown(f"```turtle\n{ontology_ttl}\n```"))

We can also inspect the ontology programmatically using rdflib. This parses the Turtle file and extracts the classes and properties, confirming what the ontology defines:


In [ ]:
# Parse and inspect the ontology programmatically
from rdflib import Graph, Namespace, RDF, RDFS, OWL

INS = Namespace("http://example.org/insurance#")

ont = Graph()
ont.parse(str(ONTOLOGY_DIR / "ontology.ttl"))

classes = sorted([str(s).replace(str(INS), ":") for s in ont.subjects(RDF.type, OWL.Class)])
data_props = sorted([str(s).replace(str(INS), ":") for s in ont.subjects(RDF.type, OWL.DatatypeProperty)])
obj_props = sorted([str(s).replace(str(INS), ":") for s in ont.subjects(RDF.type, OWL.ObjectProperty)])

print(f"Classes ({len(classes)}):          {', '.join(classes)}")
print(f"Data Properties ({len(data_props)}): {', '.join(data_props)}")
print(f"Object Properties ({len(obj_props)}): {', '.join(obj_props)}")

### Key Design Pattern: SCD Type 2 Coverage History

A `Policy` can have **multiple** `PolicyCoverage` records - each representing a version of the policy terms at a point in time (Slowly Changing Dimension Type 2).

```
Policy P002
  ├── Coverage v1: $75,000 | 2025-06-01 → 2025-12-31 (EXPIRED)
  │       ← gap: Jan 2026 - no active coverage
  └── Coverage v2: $90,000 | 2026-02-01 → present  (CURRENT)
```

This temporal gap is exactly what the eligibility rules will catch.

## What's Next?

Now that we have the ontology (the vocabulary), we need **instance data** (the facts) and **shapes** (the rules). In the next notebook, we'll load RDF data conforming to this ontology and validate it with SHACL.

→ Proceed to **02-Data-and-Validation**
